# LangChain 记忆系统 (Memory)

> 记忆系统让 LLM 拥有对话上下文能力

### 记忆类型：
1. **Buffer Memory** - 缓冲记忆（存储全部消息）
2. **Window Memory** - 滑动窗口记忆（存储最近K条）
3. **Summary Memory** - 摘要记忆（压缩对话历史）
4. **Entity Memory** - 实体记忆（提取关键实体）

In [2]:
import os
from typing import List, Dict, Any
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage, BaseMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from rich import print as rprint
import dotenv

dotenv.load_dotenv(override=True)

# 创建 LLM
llm = ChatOpenAI(
    model=os.getenv("MODEL_NAME"),
    openai_api_key=os.getenv("OPENAI_API_KEY"),
    openai_api_base=os.getenv("OPENAI_BASE_URL"),
    temperature=0
)

print("LLM 初始化完成")

LLM 初始化完成


## 1. 自定义记忆类

LangChain 新版中，我们使用自定义记忆类来管理对话历史

In [3]:
class ConversationBufferMemory:
    """缓冲记忆 - 存储所有对话历史"""
    
    def __init__(self, max_messages: int = 100):
        self.messages: List[BaseMessage] = []
        self.max_messages = max_messages
    
    def add_user_message(self, content: str):
        """添加用户消息"""
        self.messages.append(HumanMessage(content=content))
        self._trim()
    
    def add_ai_message(self, content: str):
        """添加 AI 消息"""
        self.messages.append(AIMessage(content=content))
        self._trim()
    
    def get_messages(self) -> List[BaseMessage]:
        """获取所有消息"""
        return self.messages
    
    def clear(self):
        """清空记忆"""
        self.messages = []
    
    def _trim(self):
        """保持消息数量在限制内"""
        if len(self.messages) > self.max_messages:
            self.messages = self.messages[-self.max_messages:]


class ConversationWindowMemory:
    """滑动窗口记忆 - 只保留最近 K 轮对话"""
    
    def __init__(self, k: int = 5):
        self.messages: List[BaseMessage] = []
        self.k = k  # 保留的轮数
    
    def add_user_message(self, content: str):
        self.messages.append(HumanMessage(content=content))
        self._trim()
    
    def add_ai_message(self, content: str):
        self.messages.append(AIMessage(content=content))
        self._trim()
    
    def get_messages(self) -> List[BaseMessage]:
        return self.messages
    
    def clear(self):
        self.messages = []
    
    def _trim(self):
        """只保留最近 k 轮（每轮 = 用户消息 + AI 消息）"""
        max_msgs = self.k * 2
        if len(self.messages) > max_msgs:
            self.messages = self.messages[-max_msgs:]


print("记忆类定义完成")

记忆类定义完成


## 2. 带记忆的对话链

In [4]:
def create_chat_chain(system_prompt: str = "你是一个有帮助的AI助手"):
    """创建带记忆的对话链"""
    prompt = ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        MessagesPlaceholder(variable_name="history"),
        ("human", "{input}")
    ])
    
    chain = prompt | llm | StrOutputParser()
    return chain


def chat_with_memory(chain, memory, user_input: str) -> str:
    """带记忆的对话"""
    # 获取历史消息
    history = memory.get_messages()
    
    # 调用链
    response = chain.invoke({
        "input": user_input,
        "history": history
    })
    
    # 保存到记忆
    memory.add_user_message(user_input)
    memory.add_ai_message(response)
    
    return response


print("对话链创建函数定义完成")

对话链创建函数定义完成


## 3. ConversationBufferMemory 测试

存储所有对话历史

In [5]:
# 创建记忆和链
buffer_memory = ConversationBufferMemory()
chain = create_chat_chain("你是一个友好的AI助手，名叫小助。记住用户说的话。")

rprint("[bold]ConversationBufferMemory 测试[/bold]\n")

# 第一轮对话
response = chat_with_memory(chain, buffer_memory, "你好，我叫小明")
rprint(f"用户: 你好，我叫小明")
rprint(f"AI: {response}\n")

# 第二轮对话
response = chat_with_memory(chain, buffer_memory, "我是一名Python开发者")
rprint(f"用户: 我是一名Python开发者")
rprint(f"AI: {response}\n")

# 第三轮对话（测试记忆）
response = chat_with_memory(chain, buffer_memory, "你还记得我的名字吗？")
rprint(f"用户: 你还记得我的名字吗？")
rprint(f"AI: {response}\n")

# 查看记忆内容
rprint("[cyan]记忆内容:[/cyan]")
for msg in buffer_memory.get_messages():
    role = "用户" if isinstance(msg, HumanMessage) else "AI"
    rprint(f"  {role}: {msg.content[:50]}...")

ConversationBufferMemory 测试

用户: 你好，我叫小明

AI: 你好，小明！很高兴认识你 😊 
我是小助，有什么我可以帮助你的吗？无论是解答问题、聊天还是其他需求，都可以告诉我哦～

用户: 我是一名Python开发者

AI: 太好了！Python 开发者 👨‍💻

Python 是一门非常强大且应用广泛的语言，无论是 Web 开发、数据分析、人工智能、自动化脚本还是其他领域都很出色。

你平时主要用 Python 做什么方向的开发呢？比如：

- **Web 开发**（Django、Flask、FastAPI）
- **数据科学 / 机器学习**（Pandas、NumPy、TensorFlow、PyTorch）
- **自动化 / 爬虫**（Selenium、Scrapy）
- **其他领域**

另外，如果你有任何 Python 相关的问题——比如代码调试、优化、架构设计、或者想找最佳实践，都可以随时问我！我很乐意帮忙 
😄

...

AI: 你好，小明！很高兴认识你 😊 我是小助，有什么我可以帮助你的吗？无论是解答问题、聊天还是其他需求，都...

用户: 我是一名Python开发者...

AI: 太好了！Python 开发者 👨‍💻

Python 是一门非常强大且应用广泛的语言，无论是 Web...

用户: 你还记得我的名字吗？...

AI: 当然记得！你叫 **小明** 😊

你是 Python 开发者，对吧？有什么我可以帮你的吗？...

## 4. ConversationWindowMemory 测试

只保留最近 K 轮对话

In [6]:
# 创建滑动窗口记忆（只保留最近2轮）
window_memory = ConversationWindowMemory(k=2)
chain = create_chat_chain("你是一个友好的AI助手。")

rprint("[bold]ConversationWindowMemory 测试 (k=2)[/bold]\n")

# 进行多轮对话
topics = [
    "我叫小明",
    "我在北京工作",
    "我喜欢Python编程",
    "我养了一只猫叫咪咪",
]

for topic in topics:
    response = chat_with_memory(chain, window_memory, topic)
    rprint(f"输入: {topic}")
    rprint(f"AI: {response[:80]}...\n")

# 测试记忆（最早的信息应该被遗忘）
response = chat_with_memory(chain, window_memory, "你还记得我的名字吗？")
rprint(f"测试记忆: 你还记得我的名字吗？")
rprint(f"AI: {response}")
rprint("[dim](由于窗口大小为2，最早的名字信息可能已被遗忘)[/dim]\n")

# 查看当前记忆
rprint("[cyan]当前记忆窗口:[/cyan]")
for msg in window_memory.get_messages():
    role = "用户" if isinstance(msg, HumanMessage) else "AI"
    rprint(f"  {role}: {msg.content[:50]}...")

ConversationWindowMemory 测试 (k=2)

输入: 我叫小明

AI: 你好，小明！很高兴认识你！😊 有什么我可以帮你的吗？...

输入: 我在北京工作

AI: 小明，你好！北京是一座充满活力和机会的城市，工作一定很忙碌吧？😊

作为首都，北京有很多独特的地方——无论是历史底蕴、文化氛围，还是现代职场的快节奏。如果有什么...

...

当前记忆窗口:

用户: 我养了一只猫叫咪咪...

AI: 哇，养猫太幸福了！🐱

咪咪是什么品种的呀？布偶、英短、橘猫、还是小田园？

在北京养猫的话，平时上...

用户: 你还记得我的名字吗？...

AI: 啊，抱歉！我回头看了看我们的对话，你好像还没有告诉过我你的名字呢 😅

是我眼花漏看了，还是你之前在...

## 5. 手动管理记忆

直接操作记忆的保存和加载

In [ ]:
# 手动管理记忆
memory = ConversationBufferMemory()

rprint("[bold]手动管理记忆[/bold]\n")

# 手动添加对话历史
memory.add_user_message("你好，我叫小明")
memory.add_ai_message("你好小明！很高兴认识你。")

memory.add_user_message("我在学习LangChain")
memory.add_ai_message("LangChain是一个很好的AI框架，有什么问题可以问我。")

# 查看记忆
rprint("[cyan]记忆内容:[/cyan]")
for msg in memory.get_messages():
    role = "用户" if isinstance(msg, HumanMessage) else "AI"
    rprint(f"  {role}: {msg.content}")

# 清空记忆
memory.clear()
rprint("\n[yellow]记忆已清空[/yellow]")
rprint(f"清空后消息数: {len(memory.get_messages())}")

## 6. 带记忆的完整对话示例

In [ ]:
# 创建完整的对话系统
memory = ConversationWindowMemory(k=5)
chain = create_chat_chain("""你是一个专业的旅游顾问助手，名叫「小旅」。

你的能力：
1. 查询目的地信息
2. 规划行程
3. 推荐住宿和美食
4. 提供旅行建议

请记住用户的需求，提供个性化的建议。""")

rprint("[bold]旅游助手对话示例[/bold]\n")

# 模拟对话
conversations = [
    "我想去北京旅游",
    "3天，2个人",
    "预算5000左右",
    "我们喜欢历史文化",
    "帮我规划一下行程",
]

for msg in conversations:
    response = chat_with_memory(chain, memory, msg)
    rprint(f"[cyan]用户:[/cyan] {msg}")
    rprint(f"[green]小旅:[/green] {response[:150]}...\n")

## 7. 记忆类型对比

In [ ]:
rprint("""[bold]记忆类型对比[/bold]

| 记忆类型 | 存储方式 | Token消耗 | 适用场景 |
|---------|---------|----------|----------|
| BufferMemory | 全部消息 | 高 | 短对话 |
| WindowMemory | 最近K条 | 中 | 中等对话 |
| SummaryMemory | 摘要 | 低 | 长对话 |
| EntityMemory | 实体提取 | 中 | 信息密集对话 |

[bold cyan]选择建议:[/bold cyan]
- 短对话 -> BufferMemory
- 长对话 -> SummaryMemory
- 需要记住关键信息 -> EntityMemory
- 平衡场景 -> WindowMemory
""")

## 8. 持久化记忆（文件存储）

In [ ]:
import json
from pathlib import Path
from datetime import datetime

class FileMemory:
    """文件持久化记忆"""
    
    def __init__(self, filepath: str):
        self.filepath = Path(filepath)
        self.messages = []
        self._load()
    
    def _load(self):
        if self.filepath.exists():
            with open(self.filepath, 'r', encoding='utf-8') as f:
                self.messages = json.load(f)
    
    def _save(self):
        with open(self.filepath, 'w', encoding='utf-8') as f:
            json.dump(self.messages, f, ensure_ascii=False, indent=2)
    
    def add(self, role: str, content: str):
        self.messages.append({
            "role": role,
            "content": content,
            "timestamp": datetime.now().isoformat()
        })
        self._save()
    
    def get_history(self) -> list:
        return self.messages
    
    def clear(self):
        self.messages = []
        self._save()

# 使用文件记忆
file_memory = FileMemory("chat_history.json")

rprint("[bold]文件持久化记忆[/bold]\n")

# 添加对话
file_memory.add("user", "你好，我叫小明")
file_memory.add("assistant", "你好小明！很高兴认识你。")
file_memory.add("user", "今天天气怎么样？")
file_memory.add("assistant", "抱歉，我无法获取实时天气信息。")

# 查看历史
rprint("[cyan]保存的历史:[/cyan]")
for item in file_memory.get_history():
    rprint(f"  [{item['role']}] {item['content']}")

rprint(f"\n[dim]历史已保存到: {file_memory.filepath}[/dim]")

## 总结

### 记忆系统核心概念

| 概念 | 说明 |
|------|------|
| 消息类型 | HumanMessage, AIMessage, SystemMessage |
| 缓冲记忆 | 存储所有历史 |
| 窗口记忆 | 只保留最近K轮 |
| 持久化 | 文件/数据库存储 |

### 使用建议
1. 短对话使用 BufferMemory
2. 长对话使用 WindowMemory 或 SummaryMemory
3. 需要持久化时使用文件或数据库存储